# Iteration 2 — Zero123++ + Depth Anything V2 Experimental Reconstruction

**Status: experimental / not yet reliably working.**

This notebook implements the intended research pipeline for single-image multi-view reconstruction. It should not be interpreted as a finished or production-ready 3D reconstruction system.

## Pipeline

`Input RGB → rembg → Zero123++ v1.2 → six synthesized views → background removal → Depth Anything V2 Small → camera-aware projection → point-cloud fusion → Poisson reconstruction → GLB`

The generated six-view layout is handled as a 2×3 grid.

In [ ]:
# Environment setup
!pip install -q diffusers transformers accelerate gradio trimesh pyvista open3d rembg onnxruntime einops kornia
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

import torch
import onnxruntime as ort

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
print(f"ONNX Runtime Version: {ort.__version__}")

In [ ]:
# ============================================================
# Experimental multi-view reconstruction engine
#
# Pipeline:
# Input
#   -> rembg
#   -> Zero123++ v1.2
#   -> 6 correct 320x320 views
#   -> background removal
#   -> Depth Anything V2 Small
#   -> camera-aware 3D projection
#   -> point-cloud fusion
#   -> Poisson mesh
#   -> GLB
#
# IMPORTANT:
# Zero123++ official output layout is 2 columns x 3 rows.
# ============================================================

import os
import gc
import math
import time

import numpy as np
import torch
import open3d as o3d
import trimesh
import rembg

from PIL import Image

from diffusers import DiffusionPipeline
from transformers import (
    AutoImageProcessor,
    AutoModelForDepthEstimation,
)


# ============================================================
# CONFIGURATION
# ============================================================

ZERO123_MODEL = "sudo-ai/zero123plus-v1.2"

DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Small-hf"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Zero123++ v1.2 official camera parameters
AZIMUTHS = [30, 90, 150, 210, 270, 330]
ELEVATIONS = [20, -10, 20, -10, 20, -10]

# v1.2 official output FOV
FOV_DEGREES = 30.0

# Normalized camera distance.
# This is NOT a real-world measurement.
CAMERA_RADIUS = 3.0

# Input processing size recommended by Zero123++
INPUT_SIZE = 320

# Number of diffusion steps
NUM_INFERENCE_STEPS = 28

# Output
OUTPUT_DIR = "/kaggle/working/multiview_3d_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# ENGINE
# ============================================================

class MultiView3DEngineV2:

    def __init__(self):

        print("=" * 60)
        print("INITIALIZING MULTI-VIEW 3D ENGINE V2")
        print("=" * 60)

        print(f"Device: {DEVICE}")
        print(f"Zero123++: {ZERO123_MODEL}")
        print(f"Depth model: {DEPTH_MODEL}")

        # ----------------------------------------------------
        # ZERO123++
        # ----------------------------------------------------

        print("\nLoading Zero123++...")

        self.pipe = DiffusionPipeline.from_pretrained(
            ZERO123_MODEL,
            custom_pipeline="sudo-ai/zero123plus-pipeline",
            torch_dtype=torch.float16,
        )

        self.pipe.to(DEVICE)

        print("Zero123++ loaded successfully.")

        # ----------------------------------------------------
        # DEPTH ANYTHING V2
        #
        # IMPORTANT:
        # Keep this on CPU initially.
        #
        # Your T4 already has ~10 GB occupied by Zero123++.
        # Keeping both models on GPU can unnecessarily increase
        # VRAM usage.
        # ----------------------------------------------------

        print("\nLoading Depth Anything V2 Small on CPU...")

        self.depth_processor = AutoImageProcessor.from_pretrained(
            DEPTH_MODEL
        )

        self.depth_model = AutoModelForDepthEstimation.from_pretrained(
            DEPTH_MODEL
        )

        self.depth_model.to("cpu")
        self.depth_model.eval()

        print("Depth Anything V2 Small loaded successfully.")

        print("\nENGINE READY.")
        print("=" * 60)


    # ========================================================
    # PREPROCESS INPUT
    # ========================================================

    def preprocess_image(self, image):

        if image is None:
            raise ValueError("Input image is None.")

        image = image.convert("RGB")

        # Remove background
        print("Removing input background...")

        image_rgba = rembg.remove(image)

        if not isinstance(image_rgba, Image.Image):
            image_rgba = Image.open(image_rgba)

        image_rgba = image_rgba.convert("RGBA")

        # Create square canvas
        width, height = image_rgba.size

        side = max(width, height)

        square = Image.new(
            "RGBA",
            (side, side),
            (0, 0, 0, 0)
        )

        x = (side - width) // 2
        y = (side - height) // 2

        square.paste(
            image_rgba,
            (x, y),
            image_rgba
        )

        # Resize
        square = square.resize(
            (INPUT_SIZE, INPUT_SIZE),
            Image.Resampling.LANCZOS
        )

        return square


    # ========================================================
    # ZERO123++ MULTI-VIEW
    # ========================================================

    def generate_multiview(self, image):

        processed = self.preprocess_image(image)

        print("\nGenerating six views with Zero123++...")

        start = time.time()

        result = self.pipe(
            image=processed.convert("RGB"),
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=4.0,
        ).images[0]

        elapsed = time.time() - start

        print(
            f"Zero123++ generation completed in "
            f"{elapsed:.2f} seconds."
        )

        return processed, result


    # ========================================================
    # CORRECT ZERO123++ VIEW SPLITTING
    #
    # OFFICIAL LAYOUT:
    #
    #   View 0 | View 1
    #   View 2 | View 3
    #   View 4 | View 5
    #
    # Each view = 320 x 320 for a 640 x 960 output.
    # ========================================================

    def split_multiview(self, multiview):

        width, height = multiview.size

        side_len = width // 2

        if height < side_len * 3:
            raise ValueError(
                f"Unexpected Zero123++ output size: "
                f"{multiview.size}"
            )

        views = []

        for y in range(0, side_len * 3, side_len):

            for x in range(0, side_len * 2, side_len):

                crop = multiview.crop(
                    (
                        x,
                        y,
                        x + side_len,
                        y + side_len
                    )
                )

                views.append(
                    crop.convert("RGB")
                )

        if len(views) != 6:
            raise RuntimeError(
                f"Expected 6 views, got {len(views)}"
            )

        print(
            f"Correctly extracted {len(views)} views "
            f"of size {views[0].size}"
        )

        return views


    # ========================================================
    # BACKGROUND REMOVAL FOR GENERATED VIEWS
    # ========================================================

    def clean_generated_views(self, views):

        cleaned = []

        print("\nRemoving backgrounds from generated views...")

        for i, view in enumerate(views):

            rgba = rembg.remove(
                view.convert("RGBA")
            )

            if not isinstance(rgba, Image.Image):
                rgba = Image.open(rgba)

            rgba = rgba.convert("RGBA")

            cleaned.append(rgba)

            print(f"View {i + 1}/6 cleaned.")

        return cleaned


    # ========================================================
    # DEPTH ANYTHING V2
    # ========================================================

    @torch.no_grad()
    def estimate_depth(self, image_rgba):

        rgb = image_rgba.convert("RGB")

        inputs = self.depth_processor(
            images=rgb,
            return_tensors="pt"
        )

        outputs = self.depth_model(
            **inputs
        )

        result = self.depth_processor.post_process_depth_estimation(
            outputs,
            target_sizes=[
                (rgb.height, rgb.width)
            ]
        )

        predicted_depth = result[0]["predicted_depth"]

        depth = predicted_depth.squeeze().cpu().numpy()

        depth = np.nan_to_num(
            depth,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        ).astype(np.float32)

        return depth


    # ========================================================
    # MASK
    # ========================================================

    def get_mask(self, image_rgba):

        rgba = np.asarray(
            image_rgba.convert("RGBA")
        )

        alpha = rgba[:, :, 3]

        mask = alpha > 32

        # Remove tiny noise using Open3D later.
        return mask


    # ========================================================
    # DEPTH VISUALIZATION
    # ========================================================

    def normalize_depth_for_display(
        self,
        depth,
        mask
    ):

        valid = depth[mask]

        if len(valid) == 0:
            return np.zeros_like(depth)

        lo = np.percentile(valid, 2)
        hi = np.percentile(valid, 98)

        if hi <= lo:
            return np.zeros_like(depth)

        normalized = (
            depth - lo
        ) / (
            hi - lo
        )

        normalized = np.clip(
            normalized,
            0,
            1
        )

        return normalized


    # ========================================================
    # CAMERA INTRINSICS
    # ========================================================

    def get_intrinsics(
        self,
        width,
        height
    ):

        # v1.2 specifies a 30-degree FOV.
        #
        # We use the horizontal FOV for fx and derive fy
        # from the image aspect ratio.

        fov_rad = math.radians(
            FOV_DEGREES
        )

        fx = (
            width / 2.0
        ) / math.tan(
            fov_rad / 2.0
        )

        fy = fx

        cx = width / 2.0
        cy = height / 2.0

        return fx, fy, cx, cy


    # ========================================================
    # CAMERA POSE
    #
    # World coordinate system:
    #
    # X = right
    # Y = up
    # Z = forward/reference
    #
    # Cameras are placed on a sphere around the object.
    # ========================================================

    def camera_pose(
        self,
        azimuth_deg,
        elevation_deg
    ):

        az = math.radians(
            azimuth_deg
        )

        el = math.radians(
            elevation_deg
        )

        # Camera center on sphere
        camera_center = np.array(
            [
                CAMERA_RADIUS
                * math.cos(el)
                * math.sin(az),

                CAMERA_RADIUS
                * math.sin(el),

                CAMERA_RADIUS
                * math.cos(el)
            ],
            dtype=np.float32
        )

        # Camera looks toward object origin
        forward = -camera_center

        forward = forward / (
            np.linalg.norm(forward)
            + 1e-8
        )

        world_up = np.array(
            [0, 1, 0],
            dtype=np.float32
        )

        # Camera right
        right = np.cross(
            forward,
            world_up
        )

        right = right / (
            np.linalg.norm(right)
            + 1e-8
        )

        # Camera up
        up = np.cross(
            right,
            forward
        )

        up = up / (
            np.linalg.norm(up)
            + 1e-8
        )

        # Camera image Y points downward
        down = -up

        # Camera -> world rotation
        R_cw = np.column_stack(
            [
                right,
                down,
                forward
            ]
        )

        return camera_center, R_cw


    # ========================================================
    # RELATIVE DEPTH -> NORMALIZED CAMERA DEPTH
    #
    # Depth Anything V2 Small produces relative inverse depth.
    # It is NOT metric depth.
    #
    # We normalize each view to a common object scale so the
    # six clouds can be fused.
    # ========================================================

    def convert_relative_depth(
        self,
        depth,
        mask
    ):

        valid = depth[mask]

        if len(valid) < 100:
            raise ValueError(
                "Not enough valid foreground pixels "
                "for depth reconstruction."
            )

        lo = np.percentile(
            valid,
            2
        )

        hi = np.percentile(
            valid,
            98
        )

        inv_depth = np.clip(
            depth,
            lo,
            hi
        )

        inv_depth = (
            inv_depth - lo
        ) / (
            hi - lo + 1e-8
        )

        # Depth Anything V2 behaves as inverse relative depth:
        # higher value = closer.
        #
        # Convert inverse depth to relative Z.
        relative_z = 1.0 / (
            0.15
            + 0.85 * inv_depth
        )

        # Normalize median foreground depth.
        median_z = np.median(
            relative_z[mask]
        )

        relative_z = (
            relative_z
            / (median_z + 1e-8)
        )

        # Put object around a normalized distance from camera.
        camera_object_depth = (
            CAMERA_RADIUS * 0.90
        )

        z = (
            relative_z
            * camera_object_depth
        )

        return z.astype(
            np.float32
        )


    # ========================================================
    # DEPTH + RGB -> 3D POINT CLOUD
    # ========================================================

    def depth_to_pointcloud(
        self,
        image_rgba,
        depth,
        view_index
    ):

        rgba = np.asarray(
            image_rgba
        )

        rgb = rgba[:, :, :3]

        mask = self.get_mask(
            image_rgba
        )

        h, w = mask.shape

        fx, fy, cx, cy = self.get_intrinsics(
            w,
            h
        )

        z = self.convert_relative_depth(
            depth,
            mask
        )

        # Pixel grid
        v, u = np.indices(
            (h, w)
        )

        valid = mask & (
            z > 0
        )

        # Downsample pixels to control memory.
        stride = 2

        valid[::stride, ::stride] = valid[
            ::stride,
            ::stride
        ]

        # Select points
        u_valid = u[valid].astype(
            np.float32
        )

        v_valid = v[valid].astype(
            np.float32
        )

        z_valid = z[valid]

        colors = (
            rgb[valid]
            .astype(np.float32)
            / 255.0
        )

        # Back-project into camera coordinates
        x = (
            (u_valid - cx)
            * z_valid
            / fx
        )

        y = (
            (v_valid - cy)
            * z_valid
            / fy
        )

        points_camera = np.column_stack(
            [
                x,
                y,
                z_valid
            ]
        )

        # Camera pose
        camera_center, R_cw = self.camera_pose(
            AZIMUTHS[view_index],
            ELEVATIONS[view_index]
        )

        # Camera -> world
        points_world = (
            points_camera @ R_cw.T
        ) + camera_center

        return points_world, colors


    # ========================================================
    # BUILD FUSED POINT CLOUD
    # ========================================================

    def build_fused_pointcloud(
        self,
        views,
        depths
    ):

        all_points = []
        all_colors = []

        print("\nBuilding six point clouds...")

        for i in range(6):

            print(
                f"Projecting view {i + 1}/6..."
            )

            points, colors = self.depth_to_pointcloud(
                views[i],
                depths[i],
                i
            )

            all_points.append(points)
            all_colors.append(colors)

        points = np.concatenate(
            all_points,
            axis=0
        )

        colors = np.concatenate(
            all_colors,
            axis=0
        )

        print(
            f"Raw fused points: {len(points):,}"
        )

        pcd = o3d.geometry.PointCloud()

        pcd.points = o3d.utility.Vector3dVector(
            points
        )

        pcd.colors = o3d.utility.Vector3dVector(
            colors
        )

        # Voxel fusion
        pcd = pcd.voxel_down_sample(
            voxel_size=0.012
        )

        print(
            f"After voxel fusion: "
            f"{len(pcd.points):,} points"
        )

        # Remove isolated points
        if len(pcd.points) > 100:

            pcd, _ = (
                pcd.remove_statistical_outlier(
                    nb_neighbors=20,
                    std_ratio=2.0
                )
            )

        print(
            f"After outlier removal: "
            f"{len(pcd.points):,} points"
        )

        return pcd


    # ========================================================
    # MESH RECONSTRUCTION
    # ========================================================

    def pointcloud_to_mesh(
        self,
        pcd
    ):

        print("\nEstimating point-cloud normals...")

        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(
                radius=0.08,
                max_nn=30
            )
        )

        pcd.normalize_normals()

        # Orient normals consistently
        try:

            pcd.orient_normals_consistent_tangent_plane(
                30
            )

        except Exception as e:

            print(
                "Normal orientation warning:",
                e
            )

        print("Running Poisson reconstruction...")

        mesh, densities = (
            o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
                pcd,
                depth=7,
                width=0,
                scale=1.1,
                linear_fit=True
            )
        )

        densities = np.asarray(
            densities
        )

        # Remove very low-density triangles/vertices
        if len(densities) > 0:

            threshold = np.percentile(
                densities,
                5
            )

            vertices_to_remove = (
                densities < threshold
            )

            mesh.remove_vertices_by_mask(
                vertices_to_remove
            )

        mesh.remove_degenerate_triangles()
        mesh.remove_duplicated_triangles()
        mesh.remove_duplicated_vertices()
        mesh.remove_non_manifold_edges()

        # Smooth slightly
        mesh = mesh.filter_smooth_taubin(
            number_of_iterations=3
        )

        # Compute normals
        mesh.compute_vertex_normals()

        print(
            f"Mesh vertices: "
            f"{len(mesh.vertices):,}"
        )

        print(
            f"Mesh triangles: "
            f"{len(mesh.triangles):,}"
        )

        return mesh


    # ========================================================
    # OPEN3D -> TRIMESH -> GLB
    # ========================================================

    def save_glb(
        self,
        mesh
    ):

        vertices = np.asarray(
            mesh.vertices
        )

        faces = np.asarray(
            mesh.triangles
        )

        vertex_colors = None

        if mesh.has_vertex_colors():

            vertex_colors = (
                np.asarray(
                    mesh.vertex_colors
                )
                * 255
            ).astype(
                np.uint8
            )

        tm = trimesh.Trimesh(
            vertices=vertices,
            faces=faces,
            vertex_colors=vertex_colors,
            process=False
        )

        output_path = os.path.join(
            OUTPUT_DIR,
            "reconstructed_3d_model.glb"
        )

        tm.export(
            output_path
        )

        print(
            "\nSaved GLB:"
        )

        print(
            output_path
        )

        return output_path


    # ========================================================
    # COMPLETE RECONSTRUCTION
    # ========================================================

    def generate_3d_mesh(
        self,
        input_image
    ):

        start_total = time.time()

        try:

            # ------------------------------------------------
            # STEP 1
            # ------------------------------------------------

            print("\n[1/6] Zero123++")

            processed, multiview = (
                self.generate_multiview(
                    input_image
                )
            )

            # ------------------------------------------------
            # STEP 2
            # ------------------------------------------------

            print("\n[2/6] Splitting six views")

            views = self.split_multiview(
                multiview
            )

            # ------------------------------------------------
            # STEP 3
            # ------------------------------------------------

            print("\n[3/6] Background removal")

            cleaned_views = (
                self.clean_generated_views(
                    views
                )
            )

            # ------------------------------------------------
            # STEP 4
            # ------------------------------------------------

            print("\n[4/6] Depth Anything V2")

            depths = []

            depth_images = []

            for i, view in enumerate(
                cleaned_views
            ):

                print(
                    f"Estimating depth "
                    f"{i + 1}/6..."
                )

                depth = self.estimate_depth(
                    view
                )

                depths.append(
                    depth
                )

                mask = self.get_mask(
                    view
                )

                display_depth = (
                    self.normalize_depth_for_display(
                        depth,
                        mask
                    )
                )

                depth_img = Image.fromarray(
                    (
                        display_depth
                        * 255
                    ).astype(
                        np.uint8
                    )
                )

                depth_images.append(
                    depth_img
                )

            # Save depth maps for inspection
            depth_dir = os.path.join(
                OUTPUT_DIR,
                "depth_maps"
            )

            os.makedirs(
                depth_dir,
                exist_ok=True
            )

            for i, depth_img in enumerate(
                depth_images
            ):

                depth_img.save(
                    os.path.join(
                        depth_dir,
                        f"depth_view_{i + 1}.png"
                    )
                )

            print(
                "Depth maps saved to:",
                depth_dir
            )

            # ------------------------------------------------
            # STEP 5
            # ------------------------------------------------

            print(
                "\n[5/6] Camera-aware "
                "point-cloud fusion"
            )

            pcd = (
                self.build_fused_pointcloud(
                    cleaned_views,
                    depths
                )
            )

            # ------------------------------------------------
            # STEP 6
            # ------------------------------------------------

            print(
                "\n[6/6] 3D mesh reconstruction"
            )

            mesh = self.pointcloud_to_mesh(
                pcd
            )

            output_path = self.save_glb(
                mesh
            )

            total_time = (
                time.time()
                - start_total
            )

            status = (
                "SUCCESS\n"
                f"Zero123++: 6 views\n"
                f"Depth: Depth Anything V2 Small\n"
                f"Point cloud: {len(pcd.points):,} points\n"
                f"Mesh: {len(mesh.vertices):,} vertices, "
                f"{len(mesh.triangles):,} triangles\n"
                f"Total time: {total_time:.1f} seconds\n"
                f"Output: {output_path}"
            )

            return output_path, status

        except Exception as e:

            import traceback

            traceback.print_exc()

            return (
                None,
                "RECONSTRUCTION ERROR:\n"
                + str(e)
            )


# ============================================================
# CREATE ENGINE
# ============================================================

engine_v2 = MultiView3DEngineV2()

print("\n")
print("ENGINE V2 READY.")

In [ ]:
# ============================================================
# Experimental end-to-end test
# ============================================================

IMAGE_PATH = os.environ.get("INPUT_IMAGE", "/kaggle/input/datasets/eshan12345/test-data-cv/images.jpeg")

image = Image.open(
    IMAGE_PATH
).convert("RGB")

output_path, status = (
    engine_v2.generate_3d_mesh(
        image
    )
)

print(status)